# AIFM PE Buyout Fund — Risk Notebook

Risk analysis for the AIFM PE Buyout Fund under AIFMD.
Strategy: European mid-market buyout, 8 portfolio companies across technology,
healthcare, industrials, consumer and energy transition sectors.

Regulatory framework:
- AIFMD: Directive 2011/61/EU
- AIFMD II: Directive 2024/927/EU
- Delegated Regulation: EU 231/2013
- Annex IV reporting: EU 231/2013, ESMA technical guidance v1.7 (July 2024)
- Annex VI stress testing: ESMA/2020/1498
- Luxembourg implementation: Law of 12 July 2013 on AIFMs (AIFM Law)
- IPEV Valuation Guidelines (International Private Equity Valuation)
- ILPA reporting standards (investor reporting best practice)

Dual UCITS/AIFM ManCo:
- CSSF Regulation 10-04 (organisational and prudential requirements)
- CSSF Regulation 22-05 (sustainability requirements, amending 10-04)

#### In this notebook

AIFM PE Buyout Fund I. Vintage 2018, EUR 200m target size, 10-year fund life.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.plot_style import ACCENT, ACCENT2, ACCENT3
from src.database import (
    get_engine, PEFund, PEPortfolioCompany,
    PEFundInvestment, PECashFlow, PENavHistory
)
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.esg_utils import build_esg_df, esg_portfolio_summary, ESG_FIELDS, ESG_THRESHOLD
from sqlalchemy.orm import Session

from src.generate_pe_fund import generate_cash_flows, COMPANIES, COMMITTED

flows = generate_cash_flows()

FUND_ID = 'AIFM_PE_Buyout'
TODAY   = '2026-05-13'
ENGINE  = get_engine()
BBG     = Bloomberg()

## 1. Load and Validate Fund Data

PE fund data is structured differently from liquid funds. There is no daily position
snapshot. Data is organised around portfolio companies, cash flows, and quarterly NAV
appraisals, stored in dedicated PE tables in SQLite.

The data flow is:

`generate_pe_fund.py` → SQLite PE tables → notebook

Tables loaded:
- `pe_funds`: fund metadata (vintage, size, investment period)
- `pe_portfolio_companies`: company master data (sector, country, stage)
- `pe_fund_investments`: fund-specific investment data (cost basis, ownership, exit)
- `pe_cash_flows`: capital calls, distributions, fees, exit proceeds
- `pe_nav_history`: quarterly NAV per portfolio company and at fund level

In production, these tables would be populated from the fund administrator system
(Investran, eFront, Allvue) rather than a synthetic generator.

In [ ]:
# MRS-65: Load and validate PE fund data
with Session(ENGINE) as session:
    fund        = session.query(PEFund).filter_by(fund_id=FUND_ID).first()
    companies   = session.query(PEPortfolioCompany).all()
    investments = session.query(PEFundInvestment).filter_by(fund_id=FUND_ID).all()
    cash_flows  = session.query(PECashFlow).filter_by(fund_id=FUND_ID).all()
    nav_history = session.query(PENavHistory).filter_by(fund_id=FUND_ID).all()

cf_df  = pd.DataFrame([{
    'date'       : cf.date,
    'company_id' : cf.company_id,
    'flow_type'  : cf.flow_type,
    'amount_eur' : cf.amount_eur,
    'description': cf.description,
} for cf in cash_flows])
cf_df['date'] = pd.to_datetime(cf_df['date'])

nav_df = pd.DataFrame([{
    'date'           : n.date,
    'company_id'     : n.company_id,
    'nav_eur'        : n.nav_eur,
    'gross_multiple' : n.gross_multiple,
    'unrealised_gain': n.unrealised_gain,
    'cost_basis_eur' : n.cost_basis_eur,
} for n in nav_history])
nav_df['date'] = pd.to_datetime(nav_df['date'])

inv_df = pd.DataFrame([{
    'company_id'     : i.company_id,
    'investment_date': i.investment_date,
    'cost_basis_eur' : i.cost_basis_eur,
    'ownership_pct'  : i.ownership_pct,
    'entry_ev_ebitda': i.entry_ev_ebitda,
    'entry_ev_sales' : i.entry_ev_sales,
    'exit_date'      : i.exit_date,
} for i in investments])

co_df = pd.DataFrame([{
    'company_id'      : c.company_id,
    'company_name'    : c.company_name,
    'sector'          : c.sector,
    'country'         : c.country,
    'investment_stage': c.investment_stage,
    'status'          : c.status,
} for c in companies])

call_flows = [f for f in flows if f['flow_type'] == 'capital_call']
fee_flows  = [f for f in flows if f['flow_type'] == 'management_fee']

total_called = sum(abs(f['amount_eur']) for f in call_flows)
total_fees   = sum(abs(f['amount_eur']) for f in fee_flows)
total_dist   = sum(f['amount_eur'] for f in flows 
                   if f['flow_type'] in ('distribution', 'exit_proceeds'))
gp_carry     = sum(f['amount_eur'] for f in flows 
                   if f['flow_type'] == 'carried_interest')


with ENGINE.connect() as conn:
    df_latest_nav = pd.read_sql(
        """SELECT company_id, nav_eur 
           FROM pe_nav_history
           WHERE fund_id = 'AIFM_PE_Buyout'
           AND company_id IS NOT NULL
           AND date = (
               SELECT MAX(date) FROM pe_nav_history nh2
               WHERE nh2.fund_id = pe_nav_history.fund_id
               AND nh2.company_id = pe_nav_history.company_id
           )""",
        conn
    )

# only active companies -- exited ones have no current NAV
active_ids = [c['company_id'] for c in COMPANIES if not c.get('exit_date')]
current_nav = df_latest_nav[df_latest_nav['company_id'].isin(active_ids)]['nav_eur'].sum()

print(f"Total capital called  : EUR {total_called:,.0f}  ({total_called/COMMITTED*100:.1f}% of committed)")
print(f"Total management fees : EUR {total_fees:,.0f}")
print(f"Total LP distributed  : EUR {total_dist:,.0f}")
print(f"GP carried interest   : EUR {gp_carry:,.0f}")
print(f"Current NAV (active companies only): EUR {current_nav:,.0f}")


In [ ]:
from src.generate_pe_fund import generate_valuation_reports, COMPANIES as COMP_LIST, compute_entry_equity_check

val_reports = generate_valuation_reports()

# ── build portfolio dataframe first ─────────────────────────────────────────
portfolio = co_df.merge(inv_df, on='company_id')
portfolio['investment_date'] = pd.to_datetime(portfolio['investment_date'])
portfolio['exit_date']       = pd.to_datetime(portfolio['exit_date'])

# ── derive exit multiple from appraisal NAV at exit date ────────────────────
def get_exit_multiple(row):
    if pd.isna(row['exit_date']):
        return None
    exit_date = row['exit_date'].strftime('%Y-%m-%d')
    cid = row['company_id']
    company_reports = [
        r for r in val_reports
        if r['company_id'] == cid and r['date'] <= exit_date
    ]
    if not company_reports:
        return None
    latest_nav = max(company_reports, key=lambda r: r['date'])['appraised_nav_eur']
    cost_basis = compute_entry_equity_check(cid)
    return latest_nav / cost_basis

portfolio['exit_multiple'] = portfolio.apply(get_exit_multiple, axis=1)

# ── display ──────────────────────────────────────────────────────────────────
portfolio_display = portfolio[[
    'company_name', 'sector', 'country', 'investment_stage',
    'status', 'investment_date', 'cost_basis_eur',
    'ownership_pct', 'entry_ev_ebitda', 'entry_ev_sales', 'exit_date', 'exit_multiple'
]].copy()

portfolio_display['investment_date'] = portfolio_display['investment_date'].dt.strftime('%Y-%m-%d')
portfolio_display['exit_date']       = portfolio_display['exit_date'].dt.strftime('%Y-%m-%d').fillna('--')
portfolio_display['exit_multiple']   = portfolio_display['exit_multiple'].map(
    lambda x: f'{x:.2f}x' if pd.notna(x) else '--')
portfolio_display['cost_basis_eur']  = portfolio_display['cost_basis_eur'].map('{:,.0f}'.format)
portfolio_display['ownership_pct']   = portfolio_display['ownership_pct'].map('{:.1f}%'.format)
portfolio_display['entry_ev_ebitda'] = portfolio_display['entry_ev_ebitda'].map(
    lambda x: f'{x:.1f}x' if pd.notna(x) else '--')
portfolio_display['entry_ev_sales']  = portfolio_display['entry_ev_sales'].map(
    lambda x: f'{x:.1f}x' if pd.notna(x) else '--')
portfolio_display.columns = [
    'Company', 'Sector', 'Country', 'Stage',
    'Status', 'Invested', 'Cost (EUR)',
    'Ownership', 'Entry EV/EBITDA', 'Entry EV/Sales', 'Exit Date', 'Exit Multiple'
]
portfolio_display.set_index('Company', inplace=True)
portfolio_display

## 2. Independent Appraisal and Valuation Reports

Unlike liquid funds where prices are observable daily in the market, PE portfolio
companies are valued quarterly by an independent appraisal firm. The ManCo cannot
self-certify valuations under AIFMD Article 19 without independent oversight.

**Who produces this data**: independent valuation firms (KPMG, Duff & Phelps,
Lincoln International) appointed by the fund's board. Data arrives quarterly as a
structured valuation report, stored here in `pe_valuation_report`.

**What the report contains:**
- Appraised NAV: equity value after deducting net debt from enterprise value
- LTM financials: revenue, EBITDA, margins from management accounts
- Enterprise value and EV/EBITDA multiple used by the appraiser
- Net debt and leverage ratio
- Discount rate (WACC) used in income approach valuations
- Covenant status and headroom
- Key risks identified by the appraiser

**Covenant types vary by company stage:**

*Buyout companies (PE_001, PE_002, PE_003, PE_004, PE_007, PE_008)*: maintenance
covenants tested quarterly against LTM EBITDA:

$$\text{Leverage ratio} = \frac{\text{Net Debt}}{EBITDA_{LTM}} \leq \text{Leverage covenant}$$

$$\text{Coverage ratio} = \frac{EBITDA_{LTM}}{\text{Interest expense}} \geq \text{Coverage covenant}$$

$$\text{Leverage headroom} = \frac{\text{Leverage covenant} - \text{Leverage ratio}}{\text{Leverage covenant}}$$

*Growth companies (PE_005 EnergyTrans, PE_006 FinTech Nordic)*: revenue and
liquidity covenants since EBITDA may be negative in early years:

$$\text{Revenue covenant}: \text{LTM Revenue} \geq \text{Minimum threshold}$$
$$\text{Cash covenant}: \text{Cash balance} \geq \text{Minimum cash}$$

For SaaS and subscription models, ARR (Annual Recurring Revenue) is tracked
as the primary growth metric alongside cash covenant.

> **Note**: unlike listed REITs where market prices update daily, direct PE
> valuations are quarterly and reflect appraiser judgment, not market transactions.
> Valuation risk (model risk) is therefore a significant component of PE risk,
> distinct from market risk in liquid portfolios.

In [ ]:
from src.database import PEValuationReport

with Session(ENGINE) as session:
    val_reports = session.query(PEValuationReport).filter_by(fund_id=FUND_ID).all()

vr_df = pd.DataFrame([{
    'company_id'        : v.company_id,
    'date'              : v.date,
    'appraised_nav_eur' : v.appraised_nav_eur,
    'ebitda_ltm_eur'    : v.ebitda_ltm_eur,
    'revenue_ltm_eur'   : v.revenue_ltm_eur,
    'ebitda_margin'     : v.ebitda_margin,
    'net_debt_eur'      : v.net_debt_eur,
    'ev_eur'            : v.ev_eur,
    'ev_ebitda'         : v.ev_ebitda,
    'leverage_ratio'    : v.leverage_ratio,
    'leverage_covenant' : v.leverage_covenant,
    'coverage_ratio'    : v.coverage_ratio,
    'coverage_covenant' : v.coverage_covenant,
    'covenant_type'     : v.covenant_type,
    'appraiser'         : v.appraiser,
    'key_risks'         : v.key_risks,
    'arr_eur'           : v.arr_eur,
} for v in val_reports])
vr_df['date'] = pd.to_datetime(vr_df['date'])

print(f"Valuation reports loaded : {len(vr_df)}")
print(f"Companies covered        : {vr_df['company_id'].nunique()}")
print(f"Date range               : {vr_df['date'].min().date()} to {vr_df['date'].max().date()}")
print()

def format_leverage(row):
    if pd.isna(row['leverage_ratio']) or row['leverage_ratio'] > 50:
        return 'N/A (neg. EBITDA)'
    return f"{row['leverage_ratio']:.2f}x"

def covenant_headroom(row):
    if pd.isna(row['leverage_covenant']) or pd.isna(row['leverage_ratio']):
        return '—'
    if row['leverage_ratio'] > 50:
        return 'BREACH'
    headroom = (row['leverage_covenant'] - row['leverage_ratio']) / row['leverage_covenant'] * 100
    if headroom < 0:
        return f'BREACH ({headroom:.1f}%)'
    elif headroom < 20:
        return f'⚠ {headroom:.1f}%'
    return f'{headroom:.1f}%'

latest = vr_df.sort_values('date').groupby('company_id').last().reset_index()
latest['leverage_display'] = latest.apply(format_leverage, axis=1)
latest['headroom']         = latest.apply(covenant_headroom, axis=1)

print(latest[['company_id', 'appraised_nav_eur', 'leverage_display', 
              'leverage_covenant', 'headroom', 'covenant_type']].to_string())

## 3. J-Curve Analysis 

The J-curve illustrates the typical return pattern of a PE fund over its life. Capital is called early (drawdowns), generating negative cumulative net cash flows in the first years. As portfolio companies are sold or recapitalised, distributions flow back and cumulative net cash flow turns positive -- the fund "exits the J".

**Key metrics computed here:**

**Net Cash Flow (period $t$)**

$$NCF_t = \text{Distributions}_t - \text{Capital Called}_t$$

**Cumulative Net Cash Flow**

$$CNCF_t = \sum_{\tau=0}^{t} NCF_{\tau}$$

The J-curve trough is the point where $CNCF_t$ is at its minimum -- typically years 2-4 for a standard buyout fund.

**Paid-In Capital**

$$PIC_t = \sum_{\tau=0}^{t} \text{Capital Called}_{\tau}$$

**Distributions to Paid-In (DPI)**

$$DPI_t = \frac{\sum_{\tau=0}^{t} \text{Distributions}_{\tau}}{PIC_t}$$

**Residual Value to Paid-In (RVPI)**

$$RVPI_t = \frac{NAV_t}{PIC_t}$$

**Total Value to Paid-In (TVPI = MOIC)**

$$TVPI_t = DPI_t + RVPI_t = \frac{\sum_{\tau=0}^{t} \text{Distributions}_{\tau} + NAV_t}{PIC_t}$$

In [ ]:
# MRS-55: J-curve visualisation

import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 1. Pull and aggregate cash flows by quarter ──────────────────────────────

with ENGINE.connect() as conn:
    df_cf = pd.read_sql(
        "SELECT date, flow_type, amount_eur FROM pe_cash_flows WHERE fund_id = 'AIFM_PE_Buyout'",
        conn
    )
    df_nav_raw = pd.read_sql(
        "SELECT company_id, date, nav_eur FROM pe_nav_history WHERE fund_id = 'AIFM_PE_Buyout'",
        conn
    )

df_cf["date"] = pd.to_datetime(df_cf["date"])
df_nav_raw["date"] = pd.to_datetime(df_nav_raw["date"])

# quarter-end label for grouping
df_cf["quarter"] = df_cf["date"].dt.to_period("Q").dt.to_timestamp("Q")
df_nav_raw["quarter"] = df_nav_raw["date"].dt.to_period("Q").dt.to_timestamp("Q")

# capital calls: amount_eur is negative in DB -- flip to positive for display
capital_called = (
    df_cf[df_cf["flow_type"] == "capital_call"]
    .groupby("quarter")["amount_eur"]
    .sum()
    .abs()
    .rename("capital_called")
)

# distributions: positive amounts
distributions = (
    df_cf[df_cf["flow_type"].isin(["distribution", "exit_proceeds"])]
    .groupby("quarter")["amount_eur"]
    .sum()
    .rename("distributions")
)

# management fees included in net cash flow but not in capital called
mgmt_fees = (
    df_cf[df_cf["flow_type"] == "management_fee"]
    .groupby("quarter")["amount_eur"]
    .sum()
    .abs()
    .rename("mgmt_fees")
)

# fund-level NAV = sum of company-level NAVs per quarter (exclude fund aggregate row)
nav_by_quarter = (
    df_nav_raw[df_nav_raw["company_id"].notna()]
    .groupby("quarter")["nav_eur"]
    .sum()
    .rename("nav")
)

# ── 2. Build master quarterly frame ─────────────────────────────────────────

quarters = nav_by_quarter.index.union(capital_called.index).union(distributions.index)

cf = pd.DataFrame(index=quarters)
cf = cf.join(capital_called).join(distributions).join(mgmt_fees).join(nav_by_quarter)
cf = cf.fillna(0)
cf = cf.sort_index()
cf.index.name = "quarter"

# net cash flow: distributions out minus capital in minus fees
cf["ncf"]      = cf["distributions"] - cf["capital_called"] - cf["mgmt_fees"]
cf["cncf"]     = cf["ncf"].cumsum()
cf["pic"]      = cf["capital_called"].cumsum()
cf["cum_dist"] = cf["distributions"].cumsum()

cf["dpi"]  = cf["cum_dist"] / cf["pic"].replace(0, float("nan"))
cf["rvpi"] = cf["nav"]      / cf["pic"].replace(0, float("nan"))
cf["tvpi"] = cf["dpi"] + cf["rvpi"]

trough_idx  = cf["cncf"].idxmin()
trough_val  = cf.loc[trough_idx, "cncf"]

print(f"Quarters in dataset : {len(cf)}")
print(f"Trough quarter      : {trough_idx.strftime('%Q-%Y')}")
print(f"Trough CNCF         : {trough_val/1e6:,.1f}M EUR")
print(f"\nLatest multiples (as of {cf.index[-1].strftime('%Q-%Y')}):")
print(f"  DPI  : {cf['dpi'].iloc[-1]:.2f}x")
print(f"  RVPI : {cf['rvpi'].iloc[-1]:.2f}x")
print(f"  TVPI : {cf['tvpi'].iloc[-1]:.2f}x")

# ── 3. Plot ──────────────────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 9),
    gridspec_kw={"height_ratios": [2, 1]},
    sharex=True
)

dates = cf.index.to_timestamp() if hasattr(cf.index, "to_timestamp") else cf.index
bar_width = 60  # days

ax1.plot(dates, cf["nav"], color="#2980b9", linewidth=1.4,
         linestyle=":", label="NAV (unrealised)")
ax1.bar(dates, -cf["capital_called"],
        width=bar_width, color="#c0392b", alpha=0.75, label="Capital called")
ax1.bar(dates, -cf["mgmt_fees"],
        width=bar_width, color="#e67e22", alpha=0.6,
        bottom=-cf["capital_called"], label="Management fees")
ax1.bar(dates, cf["distributions"],
        width=bar_width, color="#27ae60", alpha=0.75, label="Distributions")

ax1.plot(dates, cf["cncf"],
         color="#2c3e50", linewidth=2.2, marker="o", markersize=4,
         label="Cumulative NCF")
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")

ax1.annotate(
    f"Trough\n{trough_idx.strftime('%b %Y')}\n{trough_val/1e6:.1f}M",
    xy=(trough_idx, trough_val),
    xytext=(trough_idx, trough_val * 0.6),
    arrowprops=dict(arrowstyle="->", color="#c0392b"),
    fontsize=8, color="#c0392b", ha="center"
)

ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax1.set_ylabel("EUR")
ax1.set_title("J-Curve -- AIFM PE Buyout Fund", fontsize=13, fontweight="bold")
ax1.legend(loc="upper left", fontsize=8)
ax1.grid(axis="y", linestyle=":", alpha=0.5)

ax2.plot(dates, cf["tvpi"], color="#2c3e50", linewidth=2, label="TVPI")
ax2.plot(dates, cf["dpi"],  color="#27ae60", linewidth=1.6, linestyle="--", label="DPI")
ax2.plot(dates, cf["rvpi"], color="#2980b9", linewidth=1.6, linestyle=":",  label="RVPI")
ax2.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
ax2.set_ylabel("Multiple (x)")
ax2.set_xlabel("Quarter")
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(axis="y", linestyle=":", alpha=0.5)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}x"))

plt.tight_layout()
plt.show()


## 4. Exit Waterfall

When a portfolio company is sold, exit proceeds flow through a European waterfall before reaching LPs and the GP. The waterfall ensures LPs receive their capital back plus a minimum return before the GP participates in profits.

**European waterfall order:**

$$\text{1. Return of Capital} \rightarrow \text{LPs receive all contributed capital}$$

$$\text{2. Preferred Return} \rightarrow \text{LPs receive } 8\%\text{ p.a. on contributed capital}$$

$$\text{3. GP Catch-up} \rightarrow \text{100\% to GP until GP has 20\% of total profits}$$

$$\text{4. Carried Interest} \rightarrow \text{80\% LP / 20\% GP on remaining profits}$$

**Preferred return** on contributed capital $C$ over $T$ years:

$$PR = C \times \left[(1 + 8\%)^T - 1\right]$$

**GP carried interest** (after catch-up):

$$\text{Carry} = 20\% \times (\text{Gross exit} - C - PR)$$

In [ ]:
from src.generate_pe_fund import (
    generate_cash_flows, generate_valuation_reports,
    COMPANIES, HURDLE_RATE, CARRY_RATE
)
import matplotlib.patches as mpatches

val_reports = generate_valuation_reports()
flows       = generate_cash_flows(val_reports)
call_flows  = [f for f in flows if f['flow_type'] == 'capital_call']
fee_flows   = [f for f in flows if f['flow_type'] == 'management_fee']

exit_companies = [c for c in COMPANIES if c.get('exit_date')]

fig, axes = plt.subplots(1, len(exit_companies), figsize=(14, 7), sharey=False)

for ax, ex in zip(axes, exit_companies):
    cid       = ex['company_id']
    exit_date = ex['exit_date']
    name      = ex['company_name'].replace(' ', '\n')

    company_reports = [r for r in val_reports
                       if r['company_id'] == cid and r['date'] <= exit_date]
    gross = max(company_reports, key=lambda r: r['date'])['appraised_nav_eur']

    contributed = sum(abs(f['amount_eur']) for f in call_flows if f['company_id'] == cid)
    n_active = len([c for c in COMPANIES
                    if pd.Timestamp(c['investment_date']) <= pd.Timestamp(exit_date)])
    fees_paid = sum(abs(f['amount_eur']) / n_active for f in fee_flows
                    if f['date'] <= exit_date)
    total_contributed = contributed + fees_paid

    first_call = min(f['date'] for f in call_flows if f['company_id'] == cid)
    years = (pd.Timestamp(exit_date) - pd.Timestamp(first_call)).days / 365
    preferred_return = total_contributed * ((1 + HURDLE_RATE) ** years - 1)

    remaining = gross
    steps = []

    roc = min(remaining, total_contributed)
    steps.append(('Return of\nCapital', roc, '#2980b9', 'LP'))
    remaining -= roc

    if remaining > 0:
        pref = min(remaining, preferred_return)
        steps.append(('Preferred\nReturn 8%', pref, '#27ae60', 'LP'))
        remaining -= pref

    if remaining > 0:
        total_profit = gross - total_contributed
        gp_target    = total_profit * CARRY_RATE
        catchup      = min(remaining, gp_target)
        steps.append(('GP\nCatch-up', catchup, '#e67e22', 'GP'))
        remaining -= catchup

    if remaining > 0:
        steps.append(('LP 80%\nSplit', remaining * (1 - CARRY_RATE), '#2980b9', 'LP'))
        steps.append(('GP 20%\nSplit', remaining * CARRY_RATE, '#e67e22', 'GP'))

    bottom = 0
    bar_labels, bar_bottoms, bar_heights, bar_colors = [], [], [], []
    for label, amount, color, party in steps:
        bar_labels.append(label)
        bar_bottoms.append(bottom)
        bar_heights.append(amount)
        bar_colors.append(color)
        bottom += amount

    x = range(len(bar_labels))
    for i, (h, b, c) in enumerate(zip(bar_heights, bar_bottoms, bar_colors)):
        ax.bar(i, h, bottom=b, color=c, alpha=0.8, width=0.6)
        ax.text(i, b + h / 2, f'{h/1e6:.1f}M',
                ha='center', va='center', fontsize=7, color='white', fontweight='bold')

    ax.axhline(gross / 1e6, color='#2c3e50', linewidth=1.5, linestyle='--', alpha=0.7)
    ax.text(len(bar_labels) - 0.4, gross / 1e6,
            f'Gross\n{gross/1e6:.1f}M', fontsize=7, va='bottom', color='#2c3e50')
    ax.set_xticks(list(x))
    ax.set_xticklabels(bar_labels, fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.0f}M'))
    ax.set_title(f'{name}\n{exit_date}', fontsize=8, fontweight='bold')
    ax.grid(axis='y', linestyle=':', alpha=0.4)

lp_patch = mpatches.Patch(color='#2980b9', alpha=0.8, label='LP')
gp_patch = mpatches.Patch(color='#e67e22', alpha=0.8, label='GP')
fig.legend(handles=[lp_patch, gp_patch], loc='upper right', fontsize=9)
fig.suptitle('European Waterfall -- Exit Proceeds Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Fund Cash Management and Subscription Credit Facility

The fund maintains two treasury instruments:

**Cash reserve**: 8% of committed capital (EUR 28M) held at fund close, earning money market rates (3.5% p.a.). Used to pay management fees and fund expenses before LP capital is called.

**Subscription credit facility (sub line)**: revolving credit facility backed by unfunded LP commitments, maximum 15% of committed (EUR 52.5M), at EURIBOR + 150bps (5.0% p.a.). Drawn at investment date, repaid 90 days later when LP capital arrives.

$$\text{Cash interest earned} = \text{Cash balance} \times \frac{3.5\%}{4} \text{ (quarterly)}$$

$$\text{Sub line cost} = \text{Sub line drawn} \times \frac{5.0\%}{4} \text{ (quarterly)}$$

**Phalippou effect**: the sub line compresses the J-curve by delaying LP capital calls 90 days after investment. This artificially inflates IRR since capital appears to be deployed faster than it actually is from the LP perspective.

On exit, the fund retains enough cash to cover the next 4 quarters of management fees before distributing proceeds to LPs.

In [ ]:
# MRS-73: Fund cash management visualisation

with ENGINE.connect() as conn:
    df_cash = pd.read_sql(
        "SELECT date, cash_balance_eur, sub_line_drawn, net_cash_position, "
        "cumulative_interest_earned, cumulative_interest_paid "
        "FROM pe_fund_cash_management WHERE fund_id = 'AIFM_PE_Buyout' ORDER BY date",
        conn
    )

df_cash["date"] = pd.to_datetime(df_cash["date"])

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# ── panel 1: cash balance vs sub line ───────────────────────────────────────
ax1.bar(df_cash["date"], df_cash["cash_balance_eur"] / 1e6,
        width=60, color="#2980b9", alpha=0.7, label="Cash reserve")
ax1.plot(df_cash["date"], df_cash["sub_line_drawn"] / 1e6,
         color="#e67e22", linewidth=2, marker="o", markersize=4,
         label="Sub line drawn")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}M"))
ax1.set_ylabel("EUR M")
ax1.set_title("Fund Cash Management -- AIFM PE Buyout", fontsize=13, fontweight="bold")
ax1.legend(fontsize=8)
ax1.grid(axis="y", linestyle=":", alpha=0.5)

# ── panel 2: net cash position ───────────────────────────────────────────────
colors = ["#27ae60" if v >= 0 else "#c0392b"
          for v in df_cash["net_cash_position"]]
ax2.bar(df_cash["date"], df_cash["net_cash_position"] / 1e6,
        width=60, color=colors, alpha=0.8)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}M"))
ax2.set_ylabel("EUR M")
ax2.set_title("Net Cash Position (Cash reserve - Sub line)", fontsize=10)
ax2.grid(axis="y", linestyle=":", alpha=0.5)

# ── panel 3: cumulative interest earned vs paid ──────────────────────────────
ax3.plot(df_cash["date"], df_cash["cumulative_interest_earned"] / 1e6,
         color="#27ae60", linewidth=2, label="Cumulative interest earned")
ax3.plot(df_cash["date"], df_cash["cumulative_interest_paid"] / 1e6,
         color="#c0392b", linewidth=2, linestyle="--", label="Cumulative interest paid")
ax3.fill_between(df_cash["date"],
                 df_cash["cumulative_interest_earned"] / 1e6,
                 df_cash["cumulative_interest_paid"] / 1e6,
                 alpha=0.15, color="#27ae60", label="Net benefit")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}M"))
ax3.set_ylabel("EUR M")
ax3.set_xlabel("Quarter")
ax3.set_title("Cumulative Interest: Earned vs Paid", fontsize=10)
ax3.legend(fontsize=8)
ax3.grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()